In [1]:
LabelToID = {
    'O': 0,
    'B-Date': 1,
    'I-Date': 2,
    'B-Person': 3,
    'I-Person': 4,
    'B-Location': 5,
    'I-Location': 6,
    'B-Facility': 7,
    'I-Facility': 8,
    'B-Organization': 9,
    'I-Organization': 10,
    'B-Misc': 11,
    'I-Misc': 12,
    'B-Money': 13,
    'I-Money': 14,
    'B-NORP': 15,
    'I-NORP': 16,
    'B-Product': 17,
    'I-Product': 18}

In [2]:
from transformers import (
    AutoTokenizer,
    AutoModelForTokenClassification,
    TrainingArguments,
    Trainer,
    DataCollatorForTokenClassification
)
from datasets import Dataset
import numpy as np
from seqeval.metrics import classification_report

# --- Define your labels ---
label_list = LabelToID.keys()
label2id = LabelToID
id2label = {i: l for i, l in enumerate(label_list)}

# --- Load tokenizer and model ---
model_name = "distilbert-base-cased"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForTokenClassification.from_pretrained(
    model_name,
    num_labels=len(label_list),
    id2label=id2label,
    label2id=label2id
)

# --- Tokenize and align labels ---
# NER labels are per-word, but BERT tokenizes into subwords.
# We need to align labels to subword tokens.
def tokenize_and_align_labels(examples):
    tokenized = tokenizer(
        examples["tokens"],
        truncation=True,
        max_length=128,
        is_split_into_words=True  # input is already word-tokenized
    )
    all_labels = []
    for i, labels in enumerate(examples["ner_tags"]):
        word_ids = tokenized.word_ids(batch_index=i)
        aligned = []
        prev_word_id = None
        for word_id in word_ids:
            if word_id is None:
                aligned.append(-100)          # special tokens → ignored in loss
            elif word_id != prev_word_id:
                aligned.append(labels[word_id])  # first subword → real label
            else:
                aligned.append(-100)          # subsequent subwords → ignored
            prev_word_id = word_id
        all_labels.append(aligned)

    tokenized["labels"] = all_labels
    return tokenized

config.json:   0%|          | 0.00/465 [00:00<?, ?B/s]

c:\Users\matej\miniconda3\envs\ml\Lib\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\matej\.cache\huggingface\hub\models--distilbert-base-cased. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/263M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForTokenClassification LOAD REPORT from: distilbert-base-cased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [3]:
# Dataset
from datasets import load_dataset
IDtoLabel_US = {
    0:'O',
    1:'B-Misc',
    2:'B-Date',
    3:'I-Date',
    4:'B-Person',
    5:'I-Person',
    6:'B-NORP',
    7:'B-Misc',
    8:'I-Misc',
    9:'B-Misc',
    10:'I-Misc',
    11:'B-Organization',
    12:'I-Organization',
    13:'B-Misc',
    14:'I-Misc',
    15:'B-Misc',
    16:'B-Money',
    17:'I-Money',
    18:'B-Misc',
    19:'I-Misc',
    20:'B-Facility',
    21:'B-Date',
    22:'I-Misc',
    23:'B-Location',
    24:'B-Misc',
    25:'I-Misc',
    26:'I-NORP',
    27:'I-Location',
    28:'B-Product',
    29:'I-Date',
    30:'B-Misc',
    31:'I-Misc',
    32:'I-Facility',
    33:'B-Misc',
    34:'I-Product',
    35:'I-Misc',
    36:'I-Misc'
}

dataUS = load_dataset("hgissbkh/ontonotes5")
trainDataUS=dataUS['train'].to_pandas()
trainDataUS['ner_tags'] = trainDataUS['tags'].apply(lambda x: [IDtoLabel_US[i] for i in x])
trainDataUS['ner_tags'] = trainDataUS['ner_tags'].apply(lambda x: [LabelToID[i] for i in x])
trainDataUS.drop(columns='tags',inplace=True)
trainDataUS

,tokens,ner_tags
0,"[People, start, their, own, businesses, for, m...","[0, 0, 0, 0, 0, 0, 0, 0, 0]"
1,"[But, a, chance, to, fill, out, sales, -, tax,...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 11, 0, 0, 0]"
2,"[Red, tape, is, the, bugaboo, of, small, busin...","[0, 0, 0, 0, 0, 0, 0, 0, 0]"
3,"[Ironically, ,, the, person, who, wants, to, r...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."
4,"[Yet, every, business, owner, has, to, face, t...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."
...,...,...
59919,"[Probably, .]","[0, 0]"
59920,"[Why, ?]","[0, 0]"
59921,"[I, do, n't, think, they, can]","[0, 0, 0, 0, 0, 0]"
59922,"[but, I, think, they, 'll, try, .]","[0, 0, 0, 0, 0, 0, 0]"


In [4]:
dataset = Dataset.from_pandas(trainDataUS)
tokenized_dataset = dataset.map(tokenize_and_align_labels, batched=True)

Map:   0%|          | 0/59924 [00:00<?, ? examples/s]

In [5]:
# --- Example dataset (replace with your own) ---
# data = {
#     "tokens": [
#         ["Steve", "Jobs", "founded", "Apple", "in", "California", "."],
#         ["Barack", "Obama", "visited", "Berlin", "."]
#     ],
#     "ner_tags": [
#         [label2id["B-Person"], label2id["I-Person"], label2id["O"],
#          label2id["B-Organization"], label2id["O"], label2id["B-Location"], label2id["O"]],
#         [label2id["B-Person"], label2id["I-Person"], label2id["O"],
#          label2id["B-Location"], label2id["O"]]
#     ]
# }

# dataset = Dataset.from_dict(data)
# tokenized_dataset = dataset.map(tokenize_and_align_labels, batched=True)

# --- Metrics ---
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    true_labels, true_preds = [], []
    for pred_seq, label_seq in zip(predictions, labels):
        t, p = [], []
        for p_id, l_id in zip(pred_seq, label_seq):
            if l_id != -100:
                t.append(id2label[l_id])
                p.append(id2label[p_id])
        true_labels.append(t)
        true_preds.append(p)
    report = classification_report(true_labels, true_preds, output_dict=True)
    return {"f1": report["micro avg"]["f1-score"]}

# --- Training ---
args = TrainingArguments(
    output_dir="bert-ner-finetuned",
    fp16=True,
    eval_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    num_train_epochs=2,
    weight_decay=0.01,
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=tokenized_dataset,
    eval_dataset=tokenized_dataset,   # use a real val split in practice
    processing_class=tokenizer,
    data_collator=DataCollatorForTokenClassification(tokenizer),
    compute_metrics=compute_metrics
)

trainer.train()

Epoch,Training Loss,Validation Loss,F1
1,0.072195,0.054247,0.903183


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

KeyboardInterrupt: 